# 🔬 LangStack 深度研究助手 —— 交互式探索

本 Notebook 让你**单步执行**、实时观察每个节点的状态变化，直观理解：

- **LangChain**：模型/提示/输出的统一封装
- **LlamaIndex**：数据/RAG 检索层
- **LangGraph**：有状态工作流编排（plan→retrieve→reflect→write）
- **DeepAgents**：满配自主研究（进阶）

> 按顺序执行单元格即可。单元格 1-3 用 mock 数据，**不消耗 API**；单元格 4+ 需要真实 Key。

## 单元格 1：环境检查

In [ ]:
import sys
print(f'Python {sys.version}')

from dotenv import load_dotenv
load_dotenv()

def check(name, import_fn):
    try:
        import_fn()
        print(f'✅ {name} 就绪')
    except Exception as e:
        print(f'❌ {name}: {e}')

check('LangChain LLM', lambda: __import__('src.llm').llm.get_llm())
check('LangGraph 状态图', lambda: __import__('src.graph').graph.build_graph())
check('LlamaIndex 检索器', lambda: __import__('src.retriever').retriever)

## 单元格 2：LangChain 单步调用

理解「组件层」：模型、提示、输出解析如何串成管道。

In [ ]:
from src.llm import get_llm
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = get_llm()
chain = (
    ChatPromptTemplate.from_template('用一句话解释 {concept}，并给一个生活类比。')
    | llm
    | StrOutputParser()
)

concept = 'LangGraph 的 Checkpointer'
print(f'输入: {concept}\n')
result = chain.invoke({'concept': concept})
print(f'输出:\n{result}')

## 单元格 3：LlamaIndex 检索结构观察（mock，不花钱）

看看检索器**返回的数据长什么样**——这就是交给 LangGraph 的「原材料」。

In [ ]:
from unittest.mock import MagicMock

node = MagicMock()
node.text = 'LangGraph Checkpointer 在每步执行后将状态持久化到存储后端（如 PostgreSQL），支持线程级恢复和人机协同。'
node.score = 0.95
node.metadata = {'source': 'langgraph_docs.md'}

print('检索器返回结构:')
print(f'  文本: {node.text}')
print(f'  相关度: {node.score}')
print(f'  来源: {node.metadata["source"]}')

## 单元格 4：LangGraph 逐步执行（核心）

用 `stream_mode='values'` **单步观察状态流转**——这是理解 Agent 内部机制的黄金单元格。

In [ ]:
from src.graph import build_graph
from langchain_core.messages import HumanMessage

graph = build_graph()
initial_state = {'query': 'LangGraph 和 DeepAgents 怎么选？'}

print('=' * 50)
for step_name, state in graph.stream(
    initial_state,
    {'configurable': {'thread_id': 'notebook-demo'}},
    stream_mode='values'
):
    print(f'\n--- 节点 [{step_name}] 执行完毕 ---')
    for k, v in state.items():
        if k == 'messages':
            print(f'  {k}: {[ (m.content[:40]+"...") if len(m.content)>40 else m.content for m in v ]}')
        elif k == 'context':
            print(f'  {k}: {(str(v)[:80]+"...") if len(str(v))>80 else v}')
        else:
            print(f'  {k}: {v}')
print('=' * 50)

## 单元格 5：DeepAgents 内部结构观察

看看满配 Agent **内部其实也是一个 LangGraph**。需要 `pip install deepagents` 且 Python 3.11+。

In [ ]:
try:
    from deepagents import create_deep_agent
    agent = create_deep_agent(
        model='openai:gpt-4o-mini',
        tools=[],
        system_prompt='你是研究助手。',
    )
    g = agent.get_graph()
    print('DeepAgents 内部节点:', list(g.nodes.keys()))
    print('DeepAgents 内部边:')
    for e in g.edges:
        print(f'  {e.source} -> {e.target}')
except ImportError:
    print('DeepAgents 未安装。请 pip install deepagents（需 Python 3.11+）')

## 单元格 6：完整运行提示

确保 `.env` 配置了 `OPENAI_API_KEY` 或 `ANTHROPIC_API_KEY`，以及可选的 LangSmith 追踪。

命令行运行：
```bash
python -m src.main '你的问题'          # LangGraph 模式
python -m src.main '你的问题' --deep    # DeepAgents 模式
```

开启 `LANGSMITH_TRACING=true` 后，到 https://smith.langchain.com 查看完整调用链。